> **Series**: *Bayesian Modelling for Industrial Applications*, Part 2. [Part 1](https://sambaiga.github.io/pages/blog/posts/2025/10/2025-10-10-bayesion-foundation.html) learns a failure rate from a short pilot, and [Part 3](https://sambaiga.github.io/pages/blog/posts/2026/11/2026-11-21-hierarchical-bayesian-regression.html) lets each cell and each chemistry have its own curve.

## Battery Health and Its Uncertainty

A battery that has lost 20% of its capacity is at the end of its useful life, and a fleet operator needs to know when each cell will get there and how far the estimate can be trusted. A single best-fit curve answers the first question and ignores the second. A Bayesian regression answers both, because every parameter is a distribution, and the distributions carry through to the prediction.

This post builds that regression for the state of health (SoH) of lithium-ion cells with PyMC,[^pymc] using six cells of one chemistry from the CALCE laboratory. The model is judged by a test that a single fit cannot pass by luck: each cell is predicted from a model fitted to the other five, and the whole life of the held-out cell is scored. The result is mixed. The curve predicts four cells well and misses two, and the intervals of the model are too narrow for those two. That limitation is the reason for Part 3.

[^pymc]: O. Abril-Pla *et al.*, "PyMC: A modern, and comprehensive probabilistic programming framework in Python," *PeerJ Comput. Sci.*, vol. 9, p. e1516, 2023.

## The Beta Regression Model

The state of health is a capacity divided by a nominal capacity, so it lies between zero and a little above one. A Gaussian likelihood ignores that range, and extrapolated predictions can leave it. A Beta likelihood respects it,[^beta] and the Beta is parameterised by its mean $\mu$ and a precision $\phi$, where a large precision means that observations cluster tightly around the mean. The state of health $y_{ij}$ of cell $j$ at cycle $c_{ij}$ is modelled as

$$
y_{ij} \sim \mathrm{Beta}\big(\mu_{ij}\phi,\ (1-\mu_{ij})\phi\big), \qquad \mu_{ij} = \mathrm{logit}^{-1}(\eta_{ij}),
$$

where the inverse logit keeps the mean inside the unit interval. The linear predictor $\eta_{ij}$ combines a saturating exponential, which describes ageing, with the diagnostics $\mathbf{x}_{ij}$ measured in the same cycle,

$$
\eta_{ij} = \alpha - A\left(1 - e^{-\lambda c_{ij}}\right) + \mathbf{x}_{ij}^{\top}\boldsymbol{\beta}.
$$

Here $\alpha$ is the initial level on the logit scale, $A$ the amplitude of the decay, $\lambda$ its rate, with the cycle counted in thousands, and $\boldsymbol{\beta}$ the effects of the standardised diagnostics. Every cell shares the same $\alpha$, $A$, $\lambda$ and $\boldsymbol{\beta}$, which is why the model is called single-level, or completely pooled. The state of health is divided by 1.1 before it enters the likelihood, so that a cell that starts above its nominal capacity, as 4.8% of the cycles here do, stays inside the open unit interval.

[^beta]: S. Ferrari and F. Cribari-Neto, "Beta regression for modelling rates and proportions," *J. Appl. Statist.*, vol. 31, no. 7, pp. 799-815, 2004.

## Six Cells of One Chemistry

The data are the cycle-by-cycle diagnostics of the six cells of chemistry CS2 in the CALCE cycling data.[^calce] The four diagnostics are the area under the charging current, its mean, the area under the discharge voltage and its crest factor, prepared from the raw signals and standardised on the training cells only. Every eighth cycle is kept, because neighbouring cycles are almost duplicates, which leaves 563 observations with between 78 and 108 for each cell. The diagnostics are measured in the same cycle as the capacity, so the model estimates the state of health from them and does not forecast it.

[^calce]: CALCE Battery Research Group, University of Maryland, "Battery cycling data and diagnostic features." [Online]. Available: <https://ndownloader.figshare.com/files/59415941>

In [1]:
#| output: false
import os
import warnings
from importlib.metadata import version
from pathlib import Path
import tempfile

warnings.filterwarnings("ignore")
os.environ.setdefault("OMP_NUM_THREADS", "2")

import arviz as az
import numpy as np
import pandas as pd
import pymc as pm
from sklearn.preprocessing import StandardScaler

from bayes.data import calce
from bayes.plot import change, hierarchy
from bayes.plot.inline import show
from bayes.regression import evaluate
from bayes.regression import hierarchical as hier

FAST = os.environ.get("BAYES_FAST") == "1"  # a smoke test with a few draws, to check that the notebook runs
CACHE = Path("cache-fast" if FAST else "cache")
CACHE.mkdir(exist_ok=True)
FEATURES = ["charge_current_auc", "charge_current_mean", "discharge_voltage_auc", "discharge_voltage_crest"]
SETTINGS = {"draws": 40 if FAST else 500, "tune": 40 if FAST else 750, "chains": 2 if FAST else 4}
print("pymc", version("pymc"), "| arviz", version("arviz"), "| fast mode:", FAST)

pymc 5.25.1 | arviz 0.22.0 | fast mode: False


In [2]:
#| output: false
data = calce.load(Path(tempfile.gettempdir()) / "calce-battery-features.parquet", FEATURES, every=8)
cs2 = data[data["CellType"] == "CS2"].reset_index(drop=True)
CELLS = sorted(cs2["cell"].unique())
COLOURS = hierarchy.palette(["CS2"])
scaler = StandardScaler().fit(cs2[FEATURES])  # standardised on all six cells, for the parameters of the fit below only

for cell, rows in cs2.groupby("cell"):
    print(f"{cell}: {len(rows)} kept cycles, last cycle {rows['cycle'].max()}, first state of health {rows['soh'].iloc[0]:.1f}%")
print(f"{len(cs2):,} observations in {len(CELLS)} cells; SoH above 100%: {(cs2['soh'] > 100).mean():.1%} of the kept cycles")

CS2_33: 78 kept cycles, last cycle 737, first state of health 104.6%
CS2_34: 79 kept cycles, last cycle 775, first state of health 103.9%
CS2_35: 95 kept cycles, last cycle 912, first state of health 101.0%
CS2_36: 96 kept cycles, last cycle 908, first state of health 102.3%
CS2_37: 107 kept cycles, last cycle 982, first state of health 100.9%
CS2_38: 108 kept cycles, last cycle 1056, first state of health 101.3%
563 observations in 6 cells; SoH above 100%: 4.8% of the kept cycles


In [3]:
#| label: fig-curves
#| fig-cap: "State of health of the six CS2 cells against cycle. Each cell declines slowly and then falls steeply near the end, and the cells last between 737 and 1,056 cycles. The dotted line is the end-of-life threshold of 80%."
show(hierarchy.curves(cs2, [], COLOURS))

The six cells follow one pattern, shown in @fig-curves. Each starts near or just above 100% and declines slowly for most of its life, and then falls steeply near the end. The cells last between 737 and 1,056 cycles, so the longest lasts 43% more cycles than the shortest, which a single curve has to accommodate.

## Priors and the Prior Predictive Check

Each parameter has a weakly informative prior, which rules out values that no battery could show and leaves the rest to the data.[^gelmanprior] The initial level has $\mathcal{N}(2, 1)$ on the logit scale, the log rate has $\mathcal{N}(\log 5, 0.5^2)$ for a rate per thousand cycles, the amplitude has a half-normal prior with scale 1, and each diagnostic has an effect with $\mathcal{N}(0, 0.2^2)$, so that a feature has to earn a large coefficient. The precision has a log-normal prior centred on 100, which corresponds to noise of about 5 points of state of health at mid-life.

A prior predictive check shows what these choices imply before any data are seen. @fig-prior compares the state of health that the prior produces with the observed values.

[^gelmanprior]: A. Gelman, "Prior distributions for variance parameters in hierarchical models," *Bayesian Anal.*, vol. 1, no. 3, pp. 515-534, 2006.

In [4]:
#| output: false
model = hier.hierarchical_model(cs2, scaler, FEATURES, "complete")
print(sorted(variable.name for variable in model.free_RVs))

['alpha', 'amplitude', 'beta', 'loglambda', 'phi']


In [5]:
#| label: fig-prior
#| fig-cap: "Observed state of health of the six cells and the prior predictive distribution of the model. The prior is centred like the data and wider at the top, with some mass above 100%."
with model:
    prior = pm.sample_prior_predictive(draws=500, random_seed=1)
prior_soh = prior.prior_predictive["y"].to_numpy().ravel() * hier.CEILING * 100
drawn = np.random.default_rng(1).choice(prior_soh, size=min(20000, prior_soh.size), replace=False)
show(hierarchy.prior(cs2["soh"].to_numpy(), drawn))

Sampling: [alpha, amplitude, beta, loglambda, phi, y]


The prior has its 5th, 50th and 95th percentiles at 39.0%, 88.1% and 106.6%, against 41.3%, 87.8% and 99.9% in the data. The centre agrees and the prior is wider than the data at the top, with some mass above 100%, which the data also show. A weak prior should rule out the impossible and leave the rest to the data, and this one does.

## Posterior Sampling

The posterior is sampled with the No-U-Turn sampler,[^nuts] with four chains of 500 draws after 750 tuning draws and a target acceptance rate of 0.90. Every fit of the post uses the same settings, including the twelve fits of the test below, and the notebook stores each fit so that a repeat run reads them back.

[^nuts]: M. D. Hoffman and A. Gelman, "The No-U-Turn Sampler: Adaptively setting path lengths in Hamiltonian Monte Carlo," *J. Mach. Learn. Res.*, vol. 15, pp. 1593-1623, 2014.

In [6]:
#| output: false
idata = hier.sample(model, CACHE / "cs2-complete.nc", seed=2026, **SETTINGS)
report = hier.convergence(idata)
print(f"{idata.posterior.sizes['chain']} chains of {idata.posterior.sizes['draw']} draws, {report['divergences']} divergences, max R-hat {report['r_hat']:.3f}, min ESS bulk {report['ess_bulk']:.0f}, min ESS tail {report['ess_tail']:.0f}")

4 chains of 500 draws, 0 divergences, max R-hat 1.000, min ESS bulk 1126, min ESS tail 1019


### Convergence of the Chains

The fit to all six cells has no divergent transition. The worst $\hat{R}$, the disagreement between chains, is 1.000 against a common guide of 1.01,[^rhat] and the smallest bulk and tail effective sample sizes are 1,126 and 1,019. The twelve fits of the test have no divergences either, with a worst $\hat{R}$ of 1.01 and a smallest bulk effective sample size of 621. The posterior below can therefore be read as the posterior of the model.

[^rhat]: A. Vehtari, A. Gelman, D. Simpson, B. Carpenter, and P.-C. Bürkner, "Rank-normalization, folding, and localization: An improved $\hat{R}$ for assessing convergence of MCMC," *Bayesian Anal.*, vol. 16, no. 2, pp. 667-718, 2021.

## Posterior of the Parameters

The table gives the posterior mean, standard deviation and 90% interval of the four curve parameters, and @fig-effects gives the effect of each diagnostic.

In [7]:
posterior = idata.posterior.stack(sample=("chain", "draw"))
rows = {
    "Initial level (logit)": posterior["alpha"].to_numpy(),
    "Rate (per 1,000 cycles)": np.exp(posterior["loglambda"].to_numpy()),
    "Amplitude (logit)": posterior["amplitude"].to_numpy(),
    "Precision": posterior["phi"].to_numpy(),
}
summary = pd.DataFrame(
    {name: {"mean": draws.mean(), "sd": draws.std(), "5%": np.quantile(draws, 0.05), "95%": np.quantile(draws, 0.95)} for name, draws in rows.items()}
).T.round(3)
display(summary)

,mean,sd,5%,95%
Initial level (logit),1.911,0.020,1.878,1.944
"Rate (per 1,000 cycles)",1.785,0.179,1.482,2.079
Amplitude (logit),1.450,0.085,1.321,1.598
Precision,549.327,32.659,495.134,606.265


In [8]:
#| label: fig-effects
#| fig-cap: "Posterior mean and 90% interval of the effect of each standardised diagnostic on the logit of the state of health. The crest factor of the discharge voltage has the largest effect, and the charging-current features include zero."
posterior = idata.posterior.stack(sample=("chain", "draw"))
betas = posterior["beta"].to_numpy()
table = pd.DataFrame(
    {"feature": FEATURES, "mean": betas.mean(axis=1), "lower": np.quantile(betas, 0.05, axis=1), "upper": np.quantile(betas, 0.95, axis=1)}
)
show(hierarchy.effects(table))

The initial level is 1.91 on the logit scale (90% interval 1.88 to 1.94), and the decay has an amplitude of 1.45 and a rate of 1.79 per thousand cycles (1.48 to 2.08). A rate of 1.79 means that the saturating term reaches half of its amplitude after about 390 cycles, so the saturating term describes the early fall in capacity, and the effect of the diagnostics, which also change as a cell ages, is consistent with the later decline. The precision is about 550, which corresponds to noise of about two points of state of health.

Two diagnostics have effects that differ from zero. The crest factor of the discharge voltage has the largest, $-0.331$ with a 90% interval from $-0.350$ to $-0.310$, and the area under the discharge voltage has a small negative effect of $-0.028$. The two charging-current features have intervals that include zero. These effects should not be read one at a time. The diagnostics are correlated, so the model can attribute the same information to different ones, and a fit to a larger fleet in Part 3 puts most of the weight on the area under the discharge voltage instead.

## Prediction for Unseen Cells

A model that fits its training cells says little about a new one. The test therefore predicts each of the six cells from a model fitted to the other five. Every fold fits its own scaler on its five training cells, so nothing of the held-out cell reaches the model, and the whole life of the held-out cell is scored. The predictive draws include the noise of the Beta likelihood, so the 90% interval is a prediction interval for an observation and not an interval for the mean.

Each fold is scored with the error of the median in points of state of health, the coverage of the 90% interval and the interval score, which is the width of the interval plus $2/\alpha$ times the distance by which an observation falls outside it, with $\alpha = 0.1$ and averaged over the observations. The interval score is a proper score, so it cannot be lowered by an interval that is too narrow or too wide for the truth.[^gneiting] A second model without the diagnostics serves as a control: if the diagnostics carry information, removing them should hurt.

[^gneiting]: T. Gneiting and A. E. Raftery, "Strictly proper scoring rules, prediction, and estimation," *J. Amer. Statist. Assoc.*, vol. 102, no. 477, pp. 359-378, 2007.

In [9]:
#| output: false
SECTIONS = {}
for strategy in ("complete", "control"):
    SECTIONS[strategy] = evaluate.leave_one_out(cs2, FEATURES, strategy, CACHE, SETTINGS)
scores = pd.concat([SECTIONS[strategy][0] for strategy in SECTIONS], ignore_index=True)
curves = pd.concat([SECTIONS[strategy][1] for strategy in SECTIONS], ignore_index=True)

In [10]:
#| label: fig-heldout
#| fig-cap: "Median and 90% interval of the prediction for each cell from a model fitted to the other five, against the observed state of health. CS2_35 to CS2_38 are followed closely, while CS2_33 and CS2_34 are missed."
shown = curves[curves["strategy"] == "complete"].rename(columns={"soh": "observed"})
show(hierarchy.heldout(shown))

@fig-heldout shows the prediction for each held-out cell. Four cells, CS2_35 to CS2_38, are followed closely through their whole life, including the steep fall at the end. The prediction for CS2_33 lies below the observations for most of its life, and the prediction for CS2_34 lies above them after about cycle 450, so the model misjudges where on the curve each of these two cells is.

The table gives the scores. For CS2_35 to CS2_38 the error of the median is between 1.35 and 1.92 points and the intervals cover between 88.4% and 95.3% of the observations, so the model is well calibrated there, with an interval score between 7.1 and 8.0. For CS2_33 and CS2_34 the error is 5.1 and 5.3 points, the intervals cover 0.0% and 21.5%, and the interval scores are 51.4 and 74.6.

In [11]:
table = scores.pivot(index="cell", columns="strategy", values=["mae", "coverage", "interval"]).round(3)
table.columns = [f"{measure} ({'with' if strategy == 'complete' else 'without'})" for measure, strategy in table.columns]
display(table[[f"{measure} ({kind})" for measure in ("mae", "coverage", "interval") for kind in ("with", "without")]])

,mae (with),mae (without),coverage (with),coverage (without),interval (with),interval (without)
cell,,,,,,
CS2_33,5.149,3.773,0.000,0.962,51.436,23.691
CS2_34,5.348,2.150,0.215,0.987,74.643,19.469
CS2_35,1.348,5.237,0.884,0.926,8.018,31.182
CS2_36,1.486,4.467,0.938,0.896,7.422,36.918
CS2_37,1.362,5.375,0.953,0.935,7.054,23.638
CS2_38,1.916,6.511,0.907,0.741,7.775,29.709


In [12]:
#| label: fig-change
#| fig-cap: "Change in the error of the median and the interval score when the diagnostics are removed, for each held-out cell, against the model that uses them. Orange bars are worse. Removing them hurts four cells and helps two."
changes = evaluate.percent_change(
    scores, {"mae": "(a) Error of the median", "interval": "(b) Interval score"}, "control", "complete"
)
show(change.change(changes, "With the diagnostics", CELLS, ncol=2))

@fig-change compares the control with the model that uses the diagnostics, cell by cell. For the four cells that the model predicts well, removing the diagnostics raises the error of the median by 201% to 295% and the interval score by 235% to 397%, so the diagnostics carry most of the information. For CS2_33 and CS2_34 the control does better, with an interval score 54% and 74% lower. Its intervals are about 18 to 19 points wide and cover 96.2% and 98.7% of the observations, which is a wide prediction that happens to include the observations and not a better model.

Averaged over the six cells, the model with the diagnostics has an error of 2.77 points, a coverage of 65.0% and an interval score of 26.1, and the control has 4.59 points, 90.8% and 27.4. By the interval score the two models are almost equal, but they fail in different ways. The model with the diagnostics is accurate and over-confident where it is wrong, and the control is cautious everywhere and informative nowhere.

In [13]:
#| output: false
# Numbers that the text quotes. The cell prints them and is hidden from the post.
display(scores.round(3))
print(scores.groupby("strategy")[["mae", "coverage", "width", "interval"]].mean().round(3))
print({name: int((az.loo(fit, pointwise=True).pareto_k > 0.7).sum()) for name, fit in [("full", idata)]} if "log_likelihood" in idata.groups() else "no log likelihood stored")
print(summary.loc["Rate (per 1,000 cycles)"].to_dict(), "| half-time in cycles:", round(float(np.log(2) / summary.loc["Rate (per 1,000 cycles)", "mean"] * 1000)))
print("prior predictive percentiles 5/50/95:", np.percentile(prior_soh, [5, 50, 95]).round(1), "observed:", np.percentile(cs2["soh"], [5, 50, 95]).round(1))
for strategy in SECTIONS:
    for cell in CELLS:
        fit = az.from_netcdf(CACHE / f"loo-{strategy}-{cell}.nc")
        report = hier.convergence(fit)
        print(strategy, cell, report["divergences"], round(report["r_hat"], 3), round(report["ess_bulk"]))

,cell,strategy,mae,rmse,coverage,width,interval,n
0,CS2_33,complete,5.149,5.488,0.000,5.917,51.436,78
1,CS2_34,complete,5.348,6.780,0.215,3.831,74.643,79
2,CS2_35,complete,1.348,1.740,0.884,6.353,8.018,95
3,CS2_36,complete,1.486,1.858,0.938,6.470,7.422,96
4,CS2_37,complete,1.362,1.684,0.953,6.600,7.054,107
5,CS2_38,complete,1.916,2.231,0.907,6.421,7.775,108
6,CS2_33,control,3.773,5.399,0.962,18.352,23.691,78
7,CS2_34,control,2.150,3.346,0.987,19.078,19.469,79
8,CS2_35,control,5.237,7.112,0.926,18.197,31.182,95
9,CS2_36,control,4.467,7.426,0.896,18.494,36.918,96


            mae  coverage   width  interval
strategy                                   
complete  2.768     0.650   5.932    26.058
control   4.585     0.908  18.592    27.435
no log likelihood stored
{'mean': 1.785, 'sd': 0.179, '5%': 1.482, '95%': 2.079} | half-time in cycles: 388
prior predictive percentiles 5/50/95: [ 39.   88.1 106.6] observed: [41.3 87.8 99.9]
complete CS2_33 0 1.01 982
complete CS2_34 0 1.0 886


complete CS2_35 0 1.0 912
complete CS2_36 0 1.01 939
complete CS2_37 0 1.01 1115


complete CS2_38 0 1.0 895
control CS2_33 0 1.01 631
control CS2_34 0 1.0 621
control CS2_35 0 1.01 778


control CS2_36 0 1.0 706
control CS2_37 0 1.01 724
control CS2_38 0 1.0 801


## Limits of the Study

- **Cells share one curve.** The model gives every cell the same initial level, rate and amplitude, so a cell that ages differently is predicted by the average cell. This is the assumption that the test shows to fail for CS2_33 and CS2_34.
- **Six cells are few.** Every score of the test rests on one cell. A different chemistry or more cells could change which cells are hard to predict.
- **Observations are treated as independent.** Successive cycles of one cell are strongly correlated, and the Beta noise is too small to absorb that correlation, which makes the intervals narrow.
- **The inputs are contemporaneous.** The diagnostics are measured in the same cycle as the capacity, so the study estimates the state of health and does not forecast it.
- **Diagnostics are correlated.** Individual effects are not stable across fits, as the comparison with Part 3 shows, and should be read together.
- **Every eighth cycle is used.** This shortens the run and discards little, but the observations that remain are still correlated.

## Summary and Next Step

A Beta regression with a saturating decay and four diagnostics describes the state of health of the six CS2 cells closely, and its posterior is cleanly sampled. As a predictor for a new cell it is accurate for four of the six, with a coverage of 88% to 95% and an error of 1.4 to 1.9 points, and it misses the other two by about five points with intervals that almost never contain the observations. Removing the diagnostics makes the four easy cells much worse and the two hard cells better, which shows that the diagnostics carry the information and that the two hard cells differ from the others in a way the diagnostics cannot explain.

The next step follows from that. If cells differ, the model should let them differ while still learning from the whole fleet. Part 3 does this with a hierarchical model in which each cell draws its parameters from its chemistry, and tests whether that helps on cells that the model has not seen.

## Reproducibility of the Results

The notebook is in the [bayesian-modelling](https://github.com/sambaiga/bayesian-modelling) repository and runs from the first cell to the last in the `uv` environment of the repository, with PyMC 5.25, ArviZ 0.22 and lets-plot 4.11, and downloads the feature file when it first runs. The model, the sampling, the scores and the figures are functions of the `bayes` package, which has unit tests, and the cells of the notebook only call them. The sampling settings are in the first cell, and setting the environment variable `BAYES_FAST=1` runs the whole notebook with a few draws in a minute or two, to check that it works before the full run. The fits are stored in `cache/`, so a repeat of the full run reads them back.